# 0C - Carbon Timewise Panel Plots

Plots of carbon area density - all outputs of `Notebook 01-Sources-of-Carbon.ipynb`.

A dependency for the colormaps is the package `cmcrameri`.

In [ ]:
import numpy as np
import os
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
import gplately.pygplates as pygplates
import ptt
import gplately
import gplately.tools as tools
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader as shpreader
import netCDF4
import warnings
from scipy import ndimage
import glob, os
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import matplotlib.gridspec as gridspec
from slabdip import SlabDipper
import matplotlib
import pandas as pd

from cmcrameri import cm
plt.rcParams['font.family'] = 'Helvetica'

In [ ]:
from joblib import Parallel, delayed
import joblib

In [ ]:
# Don't change this: directory to input files
output_directory = "../Outputs/PanelPlots/"
os.makedirs(output_directory, exist_ok=True)

grid_directory = "../Grids/InputGrids/"
def defineGridFiles():
    #grid_directory = input_directory+"SRGrids/"
    spreadrate_filename = grid_directory+"SpreadingRate/Alfonso2024_SPREADING_RATE_grid_{:.2f}Ma.nc"
    agegrid_filename = grid_directory+"SeafloorAge/Alfonso2024_SEAFLOOR_AGE_grid_{:.2f}Ma.nc"
    return agegrid_filename, spreadrate_filename

In [ ]:
model_dir = "./Alfonso_etal_2024_modClennettMuller/"

feature_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/"
        r"*.gpml",
    )
)

#files = glob.glob("/my/data/folder/**/*.gpml", recursive=True)


rotation_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/",
        r"*.rot",
    )
)
coastlines_filename = os.path.join(
    model_dir,
    "Coastlines",
    "Clennett__etal_2020_Coastlines.gpml",
)

static_polygons = os.path.join(
    model_dir,
    "StaticPolygons/Clennett_2020_StaticPolygons.gpml"
)

model = gplately.PlateReconstruction(
    rotation_model=rotation_filenames,
    topology_features=pygplates.FeatureCollection(
        [
            i for i in pygplates.FeaturesFunctionArgument(
                feature_filenames
            ).get_features()
            if i.get_feature_type().to_qualified_string()
            != "gpml:TopologicalSlabBoundary"
        ]
        
    ),
    static_polygons=static_polygons
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=model,
    continents=coastlines_filename,
    coastlines=coastlines_filename
)


### A function to plot all lat lon ticks

In [ ]:
def latlonticks(ax):
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False,
              linewidth=1, color='gray', alpha=0.3,)

    ax.text(0.47,-0.07, '60°E', transform=ax.transAxes)
    ax.text(0.43,-0.07, '0°', transform=ax.transAxes)
    ax.text(0.33,-0.05, '60°W', transform=ax.transAxes)
    
    gl.top_labels=False
    gl.bottom_labels=False
    return


def latlonticks_60E(ax):
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False,
              linewidth=1, color='gray', alpha=0.3,)

    ax.text(0.47,-0.07, '60°E', transform=ax.transAxes)
    
    gl.top_labels=False
    gl.bottom_labels=False
    return

## Plots of sedimentary carbon storage 
for 150, 100, 50 and 0 Ma (2 x 2 map panel) ) (use colormap lipori)

In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1), (ax2), (ax3)) = plt.subplots(3,1, subplot_kw={'projection': proj}, figsize=(12, 12), dpi=150)

time_window = 10 #Myr

times = np.array([0,50,100,][::-1])

for t, ax in enumerate([ax1,ax2,ax3]):
    
    print("Time: {}Ma".format(times[t]))
    grd = gplately.Raster(
        "../Grids/Reservoirs/Sediment/mean/carbon_sediment_grid_{:04d}.nc".format(times[t])
    ).data + 0.000001 # slightly offset all values from 0 to ensure they aren't cropped by lognorm
    
    
    lognorm_cmap = cm.batlow
    
    # Sample the colormap to create an array of colors
    colors = lognorm_cmap(np.linspace(0, 1, 256))

    lognorm_cmap.set_under(colors[0])
    lognorm_cmap.set_bad('lightgrey')

    # times by 1e6 to go from Mt to t
    im = ax.imshow(
        grd*1e6, origin='lower', cmap=lognorm_cmap,
        norm=mcolors.LogNorm(vmin=1e0, vmax=1e2),
        transform=ccrs.PlateCarree(), interpolation='nearest'
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='None', alpha=0.46)
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_all_topological_sections(ax, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, color='k', zorder=8)
    latlonticks(ax)
    
plt.subplots_adjust(wspace=0.02, hspace=0.4)

cb_ax = fig.add_axes([0.4, 0.005, 0.25, 0.01])
cb = fig.colorbar(im, cax=cb_ax, orientation='horizontal', shrink=0.8, pad=0.1, extend='max')

fig.subplots_adjust(bottom=0.07, top=0.95, left=0.002, right=1.05,
                wspace=0.001, hspace=0.3)

cb.set_label(label='Carbon area density (T/m$^2$)', fontsize=12)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"sediment_carbon_mean_area_density.{}".format(out_format), dpi=300, bbox_inches='tight'
    )

In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    grd = gplately.Raster(
        
        "../Grids/Reservoirs/Serpentinite/mean/carbon_serpentinite_grid_{:04d}.nc".format(times[t])
    ).data

    ax.set_facecolor('lightgrey')
    im = ax.imshow(
        grd*1e6, origin='lower', cmap=cm.imola, 
        transform=ccrs.PlateCarree(), interpolation='nearest',
        vmax=3,
        vmin=0
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='silver', )
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_all_topological_sections(ax, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, color='k', zorder=8)
    latlonticks(ax)

gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (T/m$^2$)', extend='max')
cb.set_label(label='Carbon area density (T/m$^2$)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)

for out_format in ["pdf", "png", "svg"]:
        fig.savefig(output_directory+"serpentinite_carbon_mean_area_density_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
                   )

## Crustal carbon storage 
at 1000, 800, 700, 600, 500, 400, 300, 200, 100, 0 Ma (2 x 5 map panel) (use colormap batlow)


In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    grd = gplately.Raster(
        
        "../Grids/Reservoirs/Crust/mean/carbon_crust_grid_{:04d}.nc".format(times[t])
    ).data

    cmap = cm.batlow

    cmap.set_bad('lightgrey')

    
    im = ax.imshow(
        grd*1e6, origin='lower', cmap=cmap,
        transform=ccrs.PlateCarree(), interpolation='nearest',
        vmin=3,
        vmax=10
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='silver',  )
    gplot.plot_trenches(ax, color='w', linewidth=4)
    if times[t] == 500. or times[t] == 700.:
        gplot.plot_all_topological_sections(ax, color='dimgrey', tessellate_degrees=1)
    else:
        gplot.plot_all_topological_sections(ax, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, zorder=8)
    latlonticks(ax)

    
gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (T/m$^2$)', extend='both')
cb.set_label(label='Carbon area density (T/m$^2$)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)


for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"crust_carbon_mean_area_density_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
               )

# Crustal carbon with carbon outflux

Carbon outgassing flux along mid-ocean ridges - methodology taken from Notebook 3: 

CO2 outflux is related to the following parameters:

- $u$ Spreading rate [cm/yr]
- $d$ Compaction length (fixed?) [log10 km]
- $T$ Temperature of the mantle [deg C]
- $c$ Fertility of the mantle [wt %]
- $v$ CO2 volatile content (elevated near hotspots) [wt ppm]

These are fitted from a series of polynomial functions.

__Ridge outflux fit:__

$$
F = A + B_u (u - u_{\mathrm{ref}}) + B_d (d - d_{\mathrm{ref}}) + B_T (T - T_{\mathrm{ref}}) + B_c (c - c_{\mathrm{ref}}) + B_v (v - v_{\mathrm{ref}})
$$

> __Citations:__
> * Keller, T., Katz, R. F., & Hirschmann, M. M. (2017). Volatiles beneath mid-ocean ridges: Deep melting, channelised transport, focusing, and metasomatism. Earth and Planetary Science Letters, 464, 55–68. https://doi.org/10.1016/j.epsl.2017.02.006
> * Le Voyer, M., Kelley, K. A., Cottrell, E. & Hauri, E. H. (2017) Heterogeneity in mantle carbon content from CO2-undersaturated basalts. Nat Commun 8: 14062.

In [ ]:
# Define the ridge outflux polynomial
u_ref = 3.0
d_ref = 1.504  # 1 - 2
T_ref = 1350.0 # 1300. - 1400.
c_ref = 19.0   # 15 - 25
v_ref = 100.0 # ±54 ppm CO2 (Le Voyer et al. (2017))
delta_v_ref = 20.0

def ridge_outflux(u, d, T, c, v):
    u = np.abs(u)
    A = 0.9919
    B_u = 0.3162
    B_d = -0.3739
    B_T = 0.0089
    B_c = 0.0294
    B_v = 0.0095
    return A + B_u*(u - u_ref) + B_d*(d - d_ref) + B_T*(T - T_ref) + B_c*(c - c_ref) + B_v*(v - v_ref)

In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    grd = gplately.Raster(
        
        "../Grids/Reservoirs/Crust/mean/carbon_crust_grid_{:04d}.nc".format(times[t])
    ).data

    cmap = cm.batlow
    cmap.set_bad('lightgrey')

    im = ax.imshow(
        grd*1e6, origin='lower', cmap=cmap,
        transform=ccrs.PlateCarree(), interpolation='nearest',
        vmin=3,
        vmax=10
    )
    ax.set_title("{} Ma".format(times[t]))
    extent_globe = [-180,180,-90,90]
    
    gplot.time = times[t]
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='silver' )
    gplot.plot_trenches(ax, color='w', linewidth=4)

    
    gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
    cax1 = fig.add_axes([0.1, 0.21, 0.25, 0.01])
    cb = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (T/m$^2$)', extend='both')
    cb.set_label(label='Carbon area density (T/m$^2$)', size='large')
    cb.ax.tick_params(labelsize=13) 

    # ---------- CALCULATE MOR OUTGASSING
    ridge_data = model.tessellate_mid_ocean_ridges(
        gplot.time,
        tessellation_threshold_radians = np.radians(0.01),
        anchor_plate_id=0
    )
    
    ridge_lon = ridge_data[:,0]
    ridge_lat = ridge_data[:,1]
    ridge_vel = ridge_data[:,2]
    ridge_len = np.radians(ridge_data[:,3]) * 1e3 * pygplates.Earth.mean_radius_in_kms

    # Set 3 dimensions tentatively in case min and max are needed in future
    CO2_outflux = np.empty((3, len(ridge_vel)))
    CO2_influx = np.empty((3, len(ridge_vel)))
    
    # Half spreading rate - set "1" for mean
    v_mean = v_ref
    CO2_outflux[1] = ridge_outflux(0.5*ridge_vel, d_ref, T_ref, c_ref, v_mean) # t/m/yr  mean

    print("{}Ma: Min and max MOR CO2 outflux (MT/yr): {}, {}".format(gplot.time, min(CO2_outflux[1]), max(CO2_outflux[1])))
    
    #vmin, vmax = colour_range["Sediment"]
    sc = ax.scatter(ridge_lon, ridge_lat, c=CO2_outflux[1], cmap='autumn', vmin=0, vmax=2,

        transform=ccrs.PlateCarree(), rasterized=True,
    )

    if times[t] == 500. or times[t] == 700.:
        gplot.plot_all_topological_sections(ax, color='dimgrey', tessellate_degrees=1)
    else:
        gplot.plot_all_topological_sections(ax, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, zorder=8)
    latlonticks_60E(ax)

    ax.set_global()



gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])

cax2 = fig.add_axes([0.5, 0.21, 0.25, 0.01])
cb2 = fig.colorbar(sc, cax=cax2, orientation='horizontal', label='Mid-ocean ridge carbon outflux MtC/yr', extend='max')
cb2.set_label(label='Mid-ocean ridge carbon outflux MtC/yr', size='large')
cb2.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"MOR_outgassing_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
    )

## Lithospheric carbon storage 
at 1000, 800, 700, 600, 500, 400, 300, 200, 100, 0 Ma (2 x 5 map panel) (use colormap lapaz)


In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    grd = gplately.Raster(
        
        "../Grids/Reservoirs/Lithosphere/mean/carbon_lithosphere_grid_{:04d}.nc".format(times[t])
    ).data

    cmap = cm.lapaz

    cmap.set_bad('lightgrey')
    
    im = ax.imshow(
        grd*1e6, origin='lower', cmap=cmap, 
        transform=ccrs.PlateCarree(), interpolation='nearest',
        vmin = 3.6,
        vmax = 4.6
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='silver', alpha=0.46, )
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_all_topological_sections(ax, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, zorder=2)
    latlonticks(ax)
    
gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb=fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (T/m$^2$)', extend='max')
cb.set_label(label='Carbon area density (T/m$^2$)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)


for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"lithosphere_carbon_mean_area_density_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
    )

## Plots of slab dip (degrees)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

def import_cpt_to_cmap(filename):
    colors = []
    positions = []
    
    with open(filename) as f:
        for line in f:
            if line.startswith('#') or line.startswith('B') or line.startswith('F') or line.startswith('N'):
                continue
            
            parts = line.split()
            if len(parts) == 4:
                pos1 = float(parts[0])
                r1, g1, b1 = map(int, parts[1].split('/'))
                pos2 = float(parts[2])
                r2, g2, b2 = map(int, parts[3].split('/'))
                
                # Append the color and positions
                colors.append((r1/255, g1/255, b1/255))
                positions.append(pos1)
                colors.append((r2/255, g2/255, b2/255))
                positions.append(pos2)
    
    # Normalize positions to be in [0, 1]
    positions = np.array(positions)
    positions = (positions - positions.min()) / (positions.max() - positions.min())
    
    # Construct colormap
    cmap = LinearSegmentedColormap.from_list("custom_cpt", list(zip(positions, colors)))
    return cmap
    
fname = './age_1000-0.cpt'
custom_cmap = import_cpt_to_cmap(fname)


# Test the colormap by plotting it
plt.imshow([np.linspace(0, 1, 256)], aspect='auto', cmap=custom_cmap)
plt.colorbar()
plt.show()

In [ ]:

conv_cmap = matplotlib.colors.LinearSegmentedColormap.from_list('conv_cmap', ['white', 'mediumturquoise', 'teal', 'darkslategray', '#011C1A'])
div_cmap = matplotlib.colors.LinearSegmentedColormap.from_list('conv_cmap', ['white', 'wheat', 'darkgoldenrod', '#7D4B20', '#1F1300'])


proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    reconstruction_time = times[t]

    ax.set_title("{} Ma".format(reconstruction_time))

    extent_globe = [-180,180,-90,90]
    
    age_grid, _ = defineGridFiles()

    agegrid = gplately.Raster(age_grid.format(reconstruction_time))

    custom_cmap.set_bad('k')

    im = ax.imshow(agegrid.data, extent=extent_globe, cmap=custom_cmap, origin='lower', alpha=0.2,

               vmin=0, vmax=160, transform=ccrs.PlateCarree(), zorder=1
    )
    gplot.time = reconstruction_time
    #gplot.plot_continents(ax, facecolor='lightgrey', edgecolor='lightgrey')
    gplot.plot_plate_motion_vectors(ax, color='0.4', alpha=0.33, zorder=7, regrid_shape=20)
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='silver')
    gplot.plot_trenches(ax, zorder=9)
    gplot.plot_subduction_teeth(ax, zorder=9)

    # ---------- COLLECT TRENCH DATA FROM H2O NOTEBOOK OUTPUTS
    subduction_filename = "../H2O/Outputs/subduction_timesteps/subduction_{:04d}Ma.h5"
    df_time = pd.read_hdf(subduction_filename.format(int(reconstruction_time)))
    
    curr_subd_lon = df_time["lon"].to_numpy()
    curr_subd_lat = df_time["lat"].to_numpy()
    curr_subd_slabdip = df_time['slab_dip'].to_numpy()


    #vmin, vmax = colour_range["Sediment"]
    sc = ax.scatter(curr_subd_lon, curr_subd_lat, c=curr_subd_slabdip, cmap=cm.lipari_r, vmin=20, vmax=50,

        transform=ccrs.PlateCarree(), rasterized=True, zorder=8
    )
        
    gplot.plot_all_topological_sections(ax, linewidth=2, color='grey', tessellate_degrees=1)
    latlonticks(ax)

gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])

cax1 = fig.add_axes([0.1, 0.21, 0.25, 0.01])
cb1 = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Seafloor age (Ma)', extend='max')
cb1.set_label(label='Seafloor age (Ma)', size='large')
cb1.ax.tick_params(labelsize=13) 

cax2 = fig.add_axes([0.5, 0.21, 0.25, 0.01])
cb2 = fig.colorbar(sc, cax=cax2, orientation='horizontal', label='Slab dip (°)', extend='both')
cb2.set_label(label='Slab dip (°)', size='large')
cb2.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"slab_dip_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
    )

## Plots of subduction convergence rate and MOR spreading rate

In [ ]:

conv_cmap = matplotlib.colors.LinearSegmentedColormap.from_list('conv_cmap', ['white', 'mediumturquoise', 'teal', 'darkslategray', '#011C1A'])
div_cmap = matplotlib.colors.LinearSegmentedColormap.from_list('conv_cmap', ['white', 'wheat', 'darkgoldenrod', '#7D4B20', '#1F1300'])

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    time = times[t]
    
    # Use the 1799Ma panel at 1800Ma to account for an incomplete amount of stage rotations at the boundary time -
    # this lack of stage rotations makes ridge convergence rates non-calculatable. 
    if time == 1800:
        original_time = time
        time = 1799
        ax.set_title("{} Ma".format(original_time))
    else:
        ax.set_title("{} Ma".format(time))
        
    extent_globe = [-180,180,-90,90]

    # ---------- PLOT THE AGE GRID
    age_grid, _ = defineGridFiles()
    agegrid = gplately.Raster(age_grid.format(time))

    cmap = custom_cmap
    cmap.set_bad('k',1.)

    im = ax.imshow(agegrid.data, extent=extent_globe, cmap=cmap, origin='lower', alpha=0.2,

               vmin=0, vmax=160, transform=ccrs.PlateCarree(), zorder=1
    )

    # ---------- PLOT COASTLINES, TOPOLOGIES ETC
    gplot.time = time
    gplot.plot_plate_motion_vectors(ax, color='0.4', alpha=0.5, zorder=10, regrid_shape=20)
    gplot.plot_coastlines(ax, facecolor='0.9', edgecolor='none')
    gplot.plot_trenches(ax, zorder=9)
    gplot.plot_subduction_teeth(ax, zorder=9)

    
    # ---------- COLLECT TRENCH DATA FROM H2O NOTEBOOK OUTPUTS
    subduction_filename = "../H2O/Outputs/subduction_timesteps/subduction_{:04d}Ma.h5"
    df_time = pd.read_hdf(subduction_filename.format(int(time)))
    
    curr_subd_lon = df_time["lon"].to_numpy()
    curr_subd_lat = df_time["lat"].to_numpy()
    curr_subd_convergence = np.clip(df_time['vel'].to_numpy(), 0, 1e99)*1e2 # Start in m, convert to cm

    # ---------- CALCULATE RIDGE DATA IN-SITU
    ridge_data = model.tessellate_mid_ocean_ridges(
        time,
        tessellation_threshold_radians=np.radians(0.01),
        anchor_plate_id=0)
    
    curr_mor_lon = ridge_data[:,0]
    curr_mor_lat = ridge_data[:,1]
    curr_mor_divergence = ridge_data[:,2] # already in cm/yr

    ax.set_global()
    
    # ---------- PLOT SCATTERPLOTS OF VELOCITIES USING CUSTOM COLORMAPS

    sc = ax.scatter(curr_subd_lon, curr_subd_lat, c=curr_subd_convergence, cmap=conv_cmap, vmin=0, vmax=14,
        transform=ccrs.PlateCarree(), rasterized=True,
    )
    sc2 = ax.scatter(curr_mor_lon, curr_mor_lat, c=curr_mor_divergence, cmap=div_cmap, vmin=0, vmax=14,
        transform=ccrs.PlateCarree(), rasterized=True,
    )

    gplot.plot_all_topological_sections(ax, linewidth=2, color='grey', tessellate_degrees=1)
    latlonticks(ax)

gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])

cax1 = fig.add_axes([0.1, 0.21, 0.21, 0.01])
cb1 = fig.colorbar(sc,  cax=cax1, ticks=np.arange(0,12.51,2.5), orientation='horizontal', label='Subduction convergence rate (cm/yr)', extend='max', shrink=0.5)

cax2 = fig.add_axes([0.325, 0.21, 0.21, 0.01])
cb2 = fig.colorbar(sc2,  cax=cax2, orientation='horizontal', label='Mid-ocean ridge spreading rate (cm/yr)', extend='max')

cax3 = fig.add_axes([0.55, 0.21, 0.21, 0.01])
cb2 = fig.colorbar(im,  cax=cax3, orientation='horizontal', label='Seafloor age (Myr)', extend='max')

cb1.set_ticklabels(np.arange(0,12.51,2.5))
#cb2.set_ticklabels(np.arange(0,12.51,2.5))

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
            wspace=0.02, hspace=0.3)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"subd_convergence_1800-1100.{}".format(out_format), dpi=300, bbox_inches='tight'
    )

## Plots of seafloor spreading rate

In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    reconstruction_time = times[t]

    ax.set_title("{} Ma".format(reconstruction_time))

    ax.set_facecolor('lightgrey')

    extent_globe = [-180,180,-90,90]
    _, sr_grid = defineGridFiles()
    srgrid = gplately.Raster(sr_grid.format(reconstruction_time))

    im = ax.imshow(srgrid.data, extent=extent_globe, cmap='YlGnBu_r', origin='lower',

               vmin=0, vmax=160, transform=ccrs.PlateCarree(), zorder=1
    )
    gplot.time = reconstruction_time
    #gplot.plot_continents(ax, facecolor='lightgrey', edgecolor='none',)
    gplot.plot_plate_motion_vectors(ax, color='0.4', alpha=0.33, zorder=7, regrid_shape=20)
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='none',)
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_trenches(ax, zorder=9)
    gplot.plot_subduction_teeth(ax, zorder=9)

    # ---------- COLLECT TRENCH DATA FROM H2O NOTEBOOK OUTPUTS
    subduction_filename = "../H2O/Outputs/subduction_timesteps/subduction_{:04d}Ma.h5"
    df_time = pd.read_hdf(subduction_filename.format(int(time)))
    
    curr_subd_lon = df_time["lon"].to_numpy()
    curr_subd_lat = df_time["lat"].to_numpy()
    curr_subd_convergence = np.clip(df_time['vel'].to_numpy(), 0, 1e99)*1e2 # Start in m, convert to cm

    
    if times[t] == 0:
        gplot.plot_all_topological_sections(ax, linewidth=2, color='silver', tessellate_degrees=1)
    else:
        gplot.plot_all_topological_sections(ax, linewidth=2, color='grey', tessellate_degrees=1)
    latlonticks(ax)

# fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Seafloor age (Ma)', extend='max', shrink=0.4)
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb = fig.colorbar(im, cax=cax1, orientation='horizontal', label='Seafloor spreading rate (mm/yr)', shrink=0.8, pad=0.1, extend='max')
cb.set_label(label='Thickness (m)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.02, hspace=0.3)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"seafloor_spreading_rate_1800-0.{}".format(out_format), dpi=300, bbox_inches='tight'
)


## Plots of organic sedimentary carbon

In [ ]:
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER


proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    grd = gplately.Raster(
        
        "../Grids/Reservoirs/Organic_Sediments/mean/carbon_organic_sediments_grid_{:04d}.nc".format(times[t])
    ).data
    
    im = ax.imshow(
        grd*1e6, origin='lower', cmap=cm.batlow, vmax=5,
        transform=ccrs.PlateCarree(), interpolation='nearest'
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_continents(ax, facecolor='lightgrey', edgecolor='none',)
    #gplot.plot_coastlines(ax, facecolor='silver',  )
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_subduction_teeth(ax, color='k', zorder=8)
    
    gplot.plot_all_topological_sections(ax, linewidth=2, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    latlonticks(ax)

    
gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb=fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (T/m$^2$)', extend='max')
cb.set_label(label='Carbon area density (T/m$^2$)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.1, hspace=0.1)


for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"organic_sediments_carbon_mean_area_density.{}".format(out_format), dpi=300, bbox_inches='tight'
               )

## Plots of total sediment thickness

In [ ]:
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER


proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    grd = gplately.Raster(
        grid_directory+"TotalSediment/sed_thick_0.1d_{:.0f}.nc".format(times[t])
    ).data

    ax.set_facecolor('lightgrey')
    
    lognorm_cmap = matplotlib.cm.viridis
    lognorm_cmap.set_bad((1,1,1), alpha=0)

    # times by 1e6 to go from Mt to t
    im = ax.imshow(
        grd, origin='lower', cmap=lognorm_cmap,
        norm=mcolors.LogNorm(vmin=1e1, vmax=1e3),
        transform=ccrs.PlateCarree(), interpolation='nearest'
    )
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    gplot.plot_continents(ax, facecolor='lightgrey', edgecolor='none',)
    gplot.plot_coastlines(ax, facecolor='silver', edgecolor='none',)
    gplot.plot_trenches(ax, color='w', linewidth=4)
    gplot.plot_subduction_teeth(ax, color='k', zorder=8)
    
    gplot.plot_all_topological_sections(ax, linewidth=2, color='darkgrey', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    latlonticks(ax)

    
gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Thickness (m)', extend='max',)
cb.set_label(label='Thickness (m)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.1, hspace=0.1)

for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"total_sediment_thickness.{}".format(out_format), dpi=300, bbox_inches='tight'
               )

### Convergence rates ONLY where continental arcs are

In [ ]:

# updated carbonate platforms file
platforms_filename = "./ActiveCarbonatePlatforms_Neoproterozoic-to-Present.gpml"

# ensure all features have an active time of 0
carb_platform_fc = pygplates.FeatureCollection(platforms_filename)

In [ ]:
from scipy.spatial import cKDTree
from rasterio.features import rasterize
from rasterio.transform import from_bounds


def get_continent_polygon_coordinates(continent_polygons=None, grid=None, resX=0.2, resY=0.2):
    """
    Grids continents, and extracts coordinates
    Assumes 0.2 degree resolution by default
    """
    
    nx, ny = int(360/resX), int(180/resY)

    if grid:
        platforms_grid = gplately.Raster(grid,
            resample=(resY, resX),
        ).data
        xq, yq = np.meshgrid(np.arange(-180,180+resX,resX),
                             np.arange(-90, 90+resY, resY))
        
    elif continent_polygons:
        platforms_grid = rasterize(gplately.geometry.pygplates_to_shapely(continent_polygons),
                                   out_shape=(ny,nx),
                                   transform=from_bounds(-180, 90, 180, -90, nx, ny))

        xq, yq = np.meshgrid(np.linspace(-180,180,nx),
                             np.linspace(-90, 90, ny))

    mask_grid = platforms_grid > 0
    xcoords = xq[mask_grid]
    ycoords = yq[mask_grid]
    return xcoords, ycoords


# Get present-day subduction zone coordinates
time = 0
d_tols = [350]

def get_continental_arc_speeds(time):
    
    # reconstruct continents
    gplot.time = time
    
    agegrid_filename, spreadrate_filename = defineGridFiles()
    tessellation_threshold_radians = np.radians(0.01)

    # Get continets 
    clons, clats = get_continent_polygon_coordinates(grid=grid_directory+"/ContinentalMasks/Alfonso2024_continent_mask_{}.00Ma.nc".format(time))
    
    ###################### Subduction convergence data ########################
    # Initialise SlabDipper object, allocate it a spreading rate grid and an age grid
    dipper = SlabDipper()
    dipper._model = model
    dipper.set_age_grid_filename(agegrid_filename)
    dipper.set_spreading_rate_grid_filename(spreadrate_filename)

    subduction_df = dipper.tessellate_slab_dip(time, tessellation_threshold_radians)

    subduction_lon     = subduction_df.lon
    subduction_lat     = subduction_df.lat
    subduction_length  = subduction_df.length # m
    subd_len_sum = np.sum(subduction_length) 

    subduction_convergence = subduction_df.vel * 100 # m/yr to cm/yr
    subd_vel_mean, subd_vel_std = np.mean(subduction_convergence), np.std(subduction_convergence)

    # area subducted by trenches over 1 yr
    subd_surface_area = np.sum(subduction_convergence * subduction_length) # m^2/yr

    
    ################## K-D TREE 1: Find trenches intersecting continent polygons ############

    trench_xyz   = gplately.tools.lonlat2xyz(subduction_lon, subduction_lat, degrees=True)
    continent_xyz = gplately.tools.lonlat2xyz(clons, clats, degrees=True)
    tree = cKDTree(np.c_[continent_xyz])
    distance_to_trench, index = tree.query(np.c_[trench_xyz])
    distance_to_trench *= 6371
    
    length_per_dtol = np.zeros((len(d_tols), 1))
    
    for i, d_tol in enumerate(d_tols):
        mask_dist = distance_to_trench <= d_tol

        in_continent_trench_lons = subduction_lon[mask_dist]
        in_continent_trench_lats = subduction_lat[mask_dist]
        in_continent_trench_vel = subduction_convergence[mask_dist]
        
    ################## K-D TREE 2: Find continent arcs intersecting carbonate platforms ############
    cont_arc_xyz = gplately.tools.lonlat2xyz(in_continent_trench_lons, in_continent_trench_lats, degrees=True)
    
    # Reconstruct carbonate platforms at this time 
    reconstructed_platforms = model.reconstruct(carb_platform_fc, time)
    cp_lons, cp_lats = get_continent_polygon_coordinates(continent_polygons=reconstructed_platforms)
    carb_plat_xyz = gplately.tools.lonlat2xyz(cp_lons, cp_lats, degrees=True)
    
    carb_plat_tree = cKDTree(np.c_[carb_plat_xyz])
    distance_to_cont_arc, index = carb_plat_tree.query(np.c_[cont_arc_xyz])
    distance_to_cont_arc *= 6371
    
    for i, d_tol in enumerate(d_tols):
        mask_dist = distance_to_cont_arc <= d_tol

        carb_plat_intersecting_lons = in_continent_trench_lons[mask_dist]
        carb_plat_intersecting_lats = in_continent_trench_lats[mask_dist]
        carb_plat_intersecting_vel = in_continent_trench_vel[mask_dist]
    
    
    return carb_plat_intersecting_lons, carb_plat_intersecting_lats, carb_plat_intersecting_vel, clons, clats, cp_lons, cp_lats

In [ ]:
from scipy.spatial import cKDTree
from rasterio.features import rasterize
from rasterio.transform import from_bounds


def get_continent_polygon_coordinates(continent_polygons=None, grid=None, resX=0.2, resY=0.2):
    """
    Grids continents, and extracts coordinates
    Assumes 0.2 degree resolution by default
    """
    
    nx, ny = int(360/resX), int(180/resY)

    if grid:
        platforms_grid = gplately.Raster(grid,
            resample=(resY, resX),
        ).data
        xq, yq = np.meshgrid(np.arange(-180,180+resX,resX),
                             np.arange(-90, 90+resY, resY))
        
    elif continent_polygons:
        platforms_grid = rasterize(gplately.geometry.pygplates_to_shapely(continent_polygons),
                                   out_shape=(ny,nx),
                                   transform=from_bounds(-180, 90, 180, -90, nx, ny))

        xq, yq = np.meshgrid(np.linspace(-180,180,nx),
                             np.linspace(-90, 90, ny))

    mask_grid = platforms_grid > 0
    xcoords = xq[mask_grid]
    ycoords = yq[mask_grid]
    return xcoords, ycoords


# Get present-day subduction zone coordinates
time = 0
d_tols = [350]

"""
def get_continental_arc_speeds(time):
    gplot.time = time
    cont_grid_dir = grid_directory+"/ContinentalMasks/continent_mask_{}.0.nc".format(time)
    graster = gplately.Raster(cont_grid_dir, extent=[-180,180,-90,90])
    
    # Get continets 
    clons, clats = get_continent_polygon_coordinates(grid=cont_grid_dir)

    trench_data = model.tessellate_subduction_zones(time)

    trench_normal_azimuthal_angle = trench_data[:,7]
    trench_arcseg = trench_data[:,6]
    
    for distance_to_trench in d_tols:
        arc_distance = distance_to_trench / pygplates.Earth.mean_radius_in_kms

        dlon = arc_distance*np.sin(np.radians(trench_normal_azimuthal_angle))
        dlat = arc_distance*np.cos(np.radians(trench_normal_azimuthal_angle))
        trench_pt_lon = trench_data[:,0]
        trench_pt_lat = trench_data[:,1]
        trench_pt_vel = trench_data[:,2]
        ilon = trench_pt_lon + np.degrees(dlon)
        ilat = trench_pt_lat + np.degrees(dlat)
        
        sampled_points = graster.interpolate(ilon, ilat, method='linear', return_indices=True,)
        in_raster = []
        for i, point in enumerate(sampled_points[0]):
            if point > 0:
                in_raster.append(i)
        lat_in = []
        lon_in = []
        subd_we_count_lat = []
        subd_we_count_lon = []
        speed_we_count = []
        for index in in_raster:
            lat_in.append(ilat[index])
            lon_in.append(ilon[index])
            subd_we_count_lat.append(trench_pt_lat[index])
            subd_we_count_lon.append(trench_pt_lon[index])
            speed_we_count.append(trench_pt_vel[index])
         
        # Interpolate on carbonate platforms
        reconstructed_platforms = model.reconstruct(carb_platform_fc, time)
        nx, ny = int(360/0.2), int(180/0.2)
        cp_raster = gplately.Raster(
            rasterize(gplately.geometry.pygplates_to_shapely(reconstructed_platforms),
                                           out_shape=(ny,nx),
                                           transform=from_bounds(-180, 90, 180, -90, nx, ny))
        )
        
        xq, yq = np.meshgrid(np.linspace(-180,180,nx),
                             np.linspace(-90, 90, ny))

        mask_grid = cp_raster.data > 0
        cplon = xq[mask_grid]
        cplat = yq[mask_grid]
        
        cp_sampled_points = cp_raster.interpolate(ilon, ilat, method='linear', return_indices=True,)
        in_cp_raster = []
        for i, point in enumerate(cp_sampled_points[0]):
            if point > 0:
                in_cp_raster.append(i)
        lat_in_cp = []
        lon_in_cp = []
        subd_we_count_lat_cp = []
        subd_we_count_lon_cp = []
        speed_we_count_cp = []
        for index in in_cp_raster:
            lat_in_cp.append(ilat[index])
            lon_in_cp.append(ilon[index])
            subd_we_count_lat_cp.append(trench_pt_lat[index])
            subd_we_count_lon_cp.append(trench_pt_lon[index])
            speed_we_count_cp.append(trench_pt_vel[index])        
        
        
    return(lon_in_cp, lat_in_cp, subd_we_count_lon_cp, subd_we_count_lat_cp, speed_we_count_cp, clons, clats, cplon, cplat)
    """

In [ ]:
proj = ccrs.Mollweide(central_longitude=60)
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(5,2, subplot_kw={'projection': proj}, figsize=(15, 18), dpi=150)

time_window = 10 #Myr

times = np.array([170, 150, 130, 110, 100, 80, 60, 40, 20, 0])

for t, ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10]):
    
    
    ax.set_title("{}Ma".format(times[t]))

    gplot.time = times[t]
    
    lons, lats, speeds, clon, clat, cplon, cplat = get_continental_arc_speeds(times[t])

    im = ax.scatter(clon, clat, transform=ccrs.PlateCarree(), color='gainsboro', s=5)
    
    #gplot.plot_continents(ax, facecolor='lightgrey',  )
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='None' )
    gplot.plot_trenches(ax, color='w', linewidth=4)
    
    
    # Reconstruct carbonate platforms and plot the feature 
    reconstructed_platforms = model.reconstruct(carb_platform_fc, times[t])
    gplot.plot_feature(ax, reconstructed_platforms, color='#ABCDEF', alpha=0.5)
    
    im = ax.scatter(lons, lats, transform=ccrs.PlateCarree(), c=speeds, vmax=10, vmin=0, cmap='plasma', s=50)
    
    gplot.plot_subduction_teeth(ax, color='k', zorder=8)
    
    gplot.plot_all_topological_sections(ax, linewidth=2, color='silver', tessellate_degrees=1)
    gplot.plot_trenches(ax, label = "Trenches with polarity teeth")
    
    gplot.plot_plate_motion_vectors(ax, alpha=0.3)
    latlonticks(ax)

    
gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.01])
cb = fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Subduction convergence (cm/yr)', extend='max')
cb.set_label(label='Subduction convergence (cm/yr)', size='large')
cb.ax.tick_params(labelsize=13) 

fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                wspace=0.1, hspace=0.3)


for out_format in ["pdf", "png", "svg"]:
    fig.savefig(output_directory+"convergence_continental_arcs.{}".format(out_format), dpi=300, bbox_inches='tight'
               )